# Unify IDs Notebook

> User-friendly by default, with fallback mode for maximum compatibility.

Run this notebook top-to-bottom. It supports:
- widget mode (`ipywidgets`) when available
- text fallback mode (`input(...)`) when widget support is missing

> Recommended setup (from repository root):
```bash
cd <path-to-your-clone>/awg-utils/unify_ids
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
pip install -r requirements-notebook.txt --require-hashes
pip install -r requirements.txt --require-hashes
```

> Then open `unify_ids.ipynb` and select the `.venv` kernel in VS Code/Jupyter.

If widgets are unavailable, the notebook will still work and ask for paths via plain text prompts.

## Infrastructure

Imports, global variables, and helper functions. Run once per session before using the form below.

In [ ]:

import os
import shutil
import subprocess
import sys
from pathlib import Path

json_path = ""
svg_folder = ""
complex_id = ""
dry_run = True
verbose = True


def _normalize_user_path(raw_value):
    value = str(raw_value or "").strip().strip('"').strip("'")
    if not value:
        return ""
    value = os.path.expanduser(value)
    return os.path.abspath(value)


def _ensure_runtime_directory():
    cwd = Path.cwd()
    candidate_dirs = [cwd, cwd / "unify_ids", cwd.parent / "unify_ids"]
    for candidate in candidate_dirs:
        if (candidate / "unify_tkk_ids.py").is_file() and (candidate / "utils").is_dir():
            resolved = str(candidate.resolve())
            if str(cwd.resolve()) != resolved:
                os.chdir(resolved)
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            return resolved

    fallback = str(cwd.resolve())
    if fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback


def _run_command(args):
    try:
        result = subprocess.run(
            args, capture_output=True, text=True, check=False, timeout=8
        )
    except Exception:
        return ""
    if result.returncode == 0:
        return result.stdout.strip()
    return ""


def _json_picker_commands():
    if sys.platform == "darwin":
        script = 'POSIX path of (choose file with prompt "Select JSON file")'
        return [["osascript", "-e", script]]
    if sys.platform.startswith("linux"):
        return [
            [
                "zenity",
                "--file-selection",
                "--title",
                "Select JSON file",
                "--file-filter=*.json",
            ],
            ["kdialog", "--getopenfilename", os.getcwd(), "*.json"],
        ]
    if os.name == "nt":
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.OpenFileDialog;"
            "$d.Filter='JSON files (*.json)|*.json|All files (*.*)|*.*';"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.FileName}"
        )
        return [["powershell", "-NoProfile", "-Command", ps]]
    return []


def _folder_picker_commands():
    if sys.platform == "darwin":
        script = 'POSIX path of (choose folder with prompt "Select SVG folder")'
        return [["osascript", "-e", script]]
    if sys.platform.startswith("linux"):
        return [
            [
                "zenity",
                "--file-selection",
                "--directory",
                "--title",
                "Select SVG folder",
            ],
            ["kdialog", "--getexistingdirectory", os.getcwd()],
        ]
    if os.name == "nt":
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.FolderBrowserDialog;"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.SelectedPath}"
        )
        return [["powershell", "-NoProfile", "-Command", ps]]
    return []


def _pick_with_commands(candidates):
    for command in candidates:
        if command and shutil.which(command[0]):
            value = _run_command(command)
            if value:
                return value
    return ""


def _choose_file_json():
    path = _normalize_user_path(_pick_with_commands(_json_picker_commands()))
    if path.lower().endswith(".json"):
        return path
    return ""


def _choose_folder():
    return _normalize_user_path(_pick_with_commands(_folder_picker_commands()))


runtime_dir = _ensure_runtime_directory()
print(f"Notebook runtime directory: {runtime_dir}")


## Form

Fill in the inputs below and click a run button to execute the desired unifier.

In [ ]:

try:
    import ipywidgets as widgets
    from IPython.display import display
    widgets_error = None
except Exception as exc:
    widgets = None
    display = None
    widgets_error = exc

if widgets is None:
    print("ipywidgets is not available. Using text-input fallback mode.")
    print("Optional UI install: pip install ipywidgets")
    print(f"Widget import error: {widgets_error}")
    json_path = _normalize_user_path(input("Path to .json file: "))
    svg_folder = _normalize_user_path(input("Path to SVG folder: "))
    complex_id = input("Complex ID (for KV Unifier, e.g. 'op25'): ").strip()
    dry_run = input("Dry run? [Y/n]: ").strip().lower() != "n"
    verbose = input("Verbose? [Y/n]: ").strip().lower() != "n"
else:
    # --- widgets ---

    json_path_text = widgets.Text(
        description="JSON:",
        placeholder="Path to .json file",
        layout=widgets.Layout(width="62%"),
    )
    svg_folder_text = widgets.Text(
        description="SVG:",
        placeholder="Path to SVG folder",
        layout=widgets.Layout(width="62%"),
    )
    complex_id_text = widgets.Text(
        description="Complex ID:",
        placeholder="Complex ID for KV Unifier (e.g. 'op25')",
        layout=widgets.Layout(width="62%"),
    )
    dry_run_checkbox = widgets.Checkbox(value=True, description="Dry run", indent=False)
    verbose_checkbox = widgets.Checkbox(value=True, description="Verbose", indent=False)

    browse_json_button = widgets.Button(description="Browse", button_style="info", layout=widgets.Layout(width="15%"))
    browse_svg_button  = widgets.Button(description="Browse", button_style="info", layout=widgets.Layout(width="15%"))
    run_tkk_button     = widgets.Button(description="▶ Run TKK", button_style="success")
    run_linkbox_button = widgets.Button(description="▶ Run LinkBox", button_style="success")
    run_kv_button      = widgets.Button(description="▶ Run KV", button_style="success")

    run_output = widgets.Output()

    # --- sync ---

    def _sync_paths(_=None):
        global json_path, svg_folder, complex_id, dry_run, verbose
        json_path = _normalize_user_path(json_path_text.value)
        svg_folder = _normalize_user_path(svg_folder_text.value)
        complex_id = complex_id_text.value.strip()
        dry_run = dry_run_checkbox.value
        verbose = verbose_checkbox.value

    # --- browse ---

    def on_browse_json(_):
        path = _choose_file_json()
        if path:
            json_path_text.value = path

    def on_browse_svg(_):
        path = _choose_folder()
        if path:
            svg_folder_text.value = path

    # --- module lists ---

    _TKK_MODS = ['unify_tkk_ids', 'utils.extraction_utils', 'utils.svg_utils', 'utils.logger_utils']
    _LB_MODS  = ['unify_link_box_ids', 'utils.extraction_utils', 'utils.svg_utils', 'utils.logger_utils']
    _KV_MODS  = ['unify_kv_ids', 'utils.logger_utils']

    # --- shared runner ---

    def _do_run(label, modules, loader, runner, extra_check=None, extra_print=None):
        for mod in modules:
            sys.modules.pop(mod, None)
        _sync_paths()
        if not json_path or not os.path.isfile(json_path):
            print("❌ Error: No valid JSON file selected")
            return
        if extra_check is not None:
            error = extra_check()
            if error:
                print(f"❌ Error: {error}")
                return
        fn, Logger = loader()
        print(f"Running {label}...")
        print(f"  JSON: {json_path}")
        if extra_print:
            extra_print()
        print(f"  dry_run={dry_run}, verbose={verbose}")
        result = runner(fn, Logger(verbose=verbose, dry_run=dry_run))
        print(f"\n✓ Finished, result = {result}")

    # --- per-unifier run functions ---

    def _do_run_tkk():
        def load():
            from unify_tkk_ids import unify_tkk_ids
            from utils.logger_utils import Logger
            return unify_tkk_ids, Logger
        _do_run(
            "TKK Unifier", _TKK_MODS, load,
            lambda fn, lg: fn(json_path, svg_folder, lg),
            extra_check=lambda: None if os.path.isdir(svg_folder) else "No valid SVG folder selected",
            extra_print=lambda: print(f"  SVG Folder: {svg_folder}"),
        )

    def _do_run_linkbox():
        def load():
            from unify_link_box_ids import unify_link_box_ids
            from utils.logger_utils import Logger
            return unify_link_box_ids, Logger
        _do_run(
            "LinkBox Unifier", _LB_MODS, load,
            lambda fn, lg: fn(json_path, svg_folder, lg),
            extra_check=lambda: None if os.path.isdir(svg_folder) else "No valid SVG folder selected",
            extra_print=lambda: print(f"  SVG Folder: {svg_folder}"),
        )

    def _do_run_kv():
        def load():
            from unify_kv_ids import unify_kv_ids
            from utils.logger_utils import Logger
            return unify_kv_ids, Logger
        _do_run(
            "KV Unifier", _KV_MODS, load,
            lambda fn, lg: fn(json_path, complex_id, lg),
            extra_check=lambda: None if complex_id else "No complex_id set — fill in the Complex ID field above (e.g. 'op25')",
            extra_print=lambda: print(f"  complex_id: {complex_id}"),
        )

    # --- wire buttons ---

    def _make_handler(fn):
        def handler(_):
            run_output.clear_output()
            with run_output:
                fn()
        return handler

    can_pick_json   = any(shutil.which(cmd[0]) for cmd in _json_picker_commands())
    can_pick_folder = any(shutil.which(cmd[0]) for cmd in _folder_picker_commands())

    browse_json_button.disabled = not can_pick_json
    browse_svg_button.disabled  = not can_pick_folder

    browse_json_button.on_click(on_browse_json)
    browse_svg_button.on_click(on_browse_svg)
    json_path_text.observe(_sync_paths, names="value")
    svg_folder_text.observe(_sync_paths, names="value")
    complex_id_text.observe(_sync_paths, names="value")
    dry_run_checkbox.observe(_sync_paths, names="value")
    verbose_checkbox.observe(_sync_paths, names="value")
    run_tkk_button.on_click(_make_handler(_do_run_tkk))
    run_linkbox_button.on_click(_make_handler(_do_run_linkbox))
    run_kv_button.on_click(_make_handler(_do_run_kv))

    # --- display ---

    display(widgets.HBox([json_path_text, browse_json_button]))
    display(widgets.HBox([svg_folder_text, browse_svg_button]))
    display(widgets.HBox([complex_id_text]))
    display(widgets.HBox([verbose_checkbox, dry_run_checkbox]))
    display(widgets.HBox([run_tkk_button, run_linkbox_button, run_kv_button]))
    display(run_output)
    _sync_paths()

    if not can_pick_json or not can_pick_folder:
        print("Compatibility note: Browse buttons are disabled because no supported file picker tool is available.")

    print("You can always type paths directly into the text boxes.")
